## Librerias

In [49]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler


## Datos

In [50]:
df = pd.read_csv('dfSegmentado.csv')

## Procesamiento

In [51]:
target = "Segmento"
X = df.drop(columns=["Segmento"])  # variables predictoras
y = df["Segmento"]                 # variable objetivo

X = X.drop(columns=["Identificador"], errors="ignore")

In [52]:
numericas = X.select_dtypes(include=["int64", "float64"]).columns
categoricas = X.select_dtypes(include=["object", "category"]).columns
print("Variables numéricas:", list(numericas))
print("Variables categóricas:", list(categoricas))

Variables numéricas: ['Edad', 'Experiencia_Laboral', 'Tamano_Familiar']
Variables categóricas: ['Genero', 'Estado_Civil', 'Graduado', 'Profesion', 'Puntuacion_Gasto']


In [53]:
X = pd.get_dummies(X, columns=categoricas, drop_first=True)

In [54]:
print(f"Después de codificar, el número de características es: {X.shape[1]}")

Después de codificar, el número de características es: 16


In [55]:
#Estandariza los datos numéricos para que tengan media = 0 y desviación estándar = 1
# Evita que variables con escalas grandes dominen sobre escalas pequeñas
scaler = StandardScaler()
X[numericas] = scaler.fit_transform(X[numericas])

## Modelo - Arbol de decision

In [56]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [57]:
dt_model = DecisionTreeClassifier(
    max_depth=5,        # control overfitting
    random_state=42
)

dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

Accuracy: 0.48451053283767037
              precision    recall  f1-score   support

           A       0.39      0.28      0.33       394
           B       0.32      0.33      0.32       372
           C       0.55      0.56      0.55       394
           D       0.61      0.72      0.66       454

    accuracy                           0.48      1614
   macro avg       0.47      0.47      0.47      1614
weighted avg       0.47      0.48      0.48      1614



## Modelo - Random Forest

In [58]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Accuracy: 0.5241635687732342
              precision    recall  f1-score   support

           A       0.42      0.44      0.43       394
           B       0.39      0.28      0.33       372
           C       0.57      0.58      0.58       394
           D       0.63      0.74      0.68       454

    accuracy                           0.52      1614
   macro avg       0.50      0.51      0.51      1614
weighted avg       0.51      0.52      0.51      1614



## Comparacion 

In [59]:
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

Decision Tree Accuracy: 0.48451053283767037
Random Forest Accuracy: 0.5241635687732342


## Optimizacion Random Forest

In [60]:
features_clave = [
    "Edad",
    "Tamano_Familiar",
    "Experiencia_Laboral",
    "Puntuacion_Gasto_Low",
    "Graduado_Yes",
    "Estado_Civil_Yes"
]

X_reducido = X[features_clave]

In [61]:
df["edad_productiva"] = (df["Edad"] >= 25) & (df["Edad"] <= 55)
df["edad_productiva"] = df["edad_productiva"].astype(int)
print("Nueva variable 'edad_productiva' creada. Ejemplo de valores:")
print(df[["Edad", "edad_productiva"]].head())

Nueva variable 'edad_productiva' creada. Ejemplo de valores:
   Edad  edad_productiva
0    22                0
1    38                1
2    67                0
3    67                0
4    40                1


In [62]:
df["ratio_exp_edad"] = df["Experiencia_Laboral"] / (df["Edad"] + 1)
print("Nueva variable 'ratio_exp_edad' creada. Ejemplo de valores:")
print(df[["Experiencia_Laboral", "Edad", "ratio_exp_edad"]].head())

Nueva variable 'ratio_exp_edad' creada. Ejemplo de valores:
   Experiencia_Laboral  Edad  ratio_exp_edad
0                  1.0    22        0.043478
1                  1.0    38        0.025641
2                  1.0    67        0.014706
3                  0.0    67        0.000000
4                  1.0    40        0.024390


In [63]:
df["carga_familiar"] = df["Tamano_Familiar"] / (df["Edad"] + 1)
print("Nueva variable 'carga_familiar' creada. Ejemplo de valores:")
print(df[["Tamano_Familiar", "Edad", "carga_familiar"]].head())

Nueva variable 'carga_familiar' creada. Ejemplo de valores:
   Tamano_Familiar  Edad  carga_familiar
0              4.0    22        0.173913
1              3.0    38        0.076923
2              1.0    67        0.014706
3              2.0    67        0.029412
4              6.0    40        0.146341


In [64]:
df["gasto_alto"] = (df["Puntuacion_Gasto"] == "High").astype(int)
print("Nueva variable 'gasto_alto' creada. Ejemplo de valores:")
print(df[["Puntuacion_Gasto", "gasto_alto"]].head())

Nueva variable 'gasto_alto' creada. Ejemplo de valores:
  Puntuacion_Gasto  gasto_alto
0              Low           0
1          Average           0
2              Low           0
3             High           1
4             High           1


In [65]:
# Reentrenar modelos con nuevas características
X = df[[
    "Edad",
    "Experiencia_Laboral",
    "Tamano_Familiar",
    "ratio_exp_edad",
    "carga_familiar",
    "edad_productiva",
    "gasto_alto"
]]

y = df["Segmento"]

In [66]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_split=5,
    random_state=42
)

In [67]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [68]:
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

In [69]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.442998760842627
              precision    recall  f1-score   support

           A       0.35      0.35      0.35       394
           B       0.29      0.11      0.16       372
           C       0.41      0.58      0.48       394
           D       0.58      0.68      0.63       454

    accuracy                           0.44      1614
   macro avg       0.41      0.43      0.40      1614
weighted avg       0.42      0.44      0.42      1614



¿Por qué empeoró?

Al reducir las variables solo a las "clave" y las nuevas creadas, eliminamos información valiosa que el primer modelo sí estaba usando (como Profesion o Estado_Civil que habías codificado con get_dummies).